# UCD — Unsupervised Crack Detection (Kaggle)

Port of the Colab `UCD.ipynb`. One step per cell.

**Before running:**
- `Settings -> Accelerator`: **GPU** (T4 x2 or P100)
- `Settings -> Internet`: **On** (needed for pip / git / wget — requires phone verification)

**Run order:** Cell 1 (installs) -> **Restart kernel** -> Cell 2 onward.

Paths: `/kaggle/temp` = scratch (not saved, use for bulk data), `/kaggle/working` = notebook
output (~20 GB cap, use for code / checkpoints / metrics), `/kaggle/input` = read-only datasets.

## 1 — Dependencies

Kaggle ships torch+CUDA preinstalled. `anomalib` can drag in a **CPU-only torch wheel** and
silently kill the GPU, so torch is pinned out of the resolver via a constraints file and
CUDA is asserted at the end. `numpy<2` needs a kernel restart to take effect.

In [ ]:
import subprocess, sys
from pathlib import Path

torch_pin = subprocess.run(
    [sys.executable, "-c",
     "import torch;print(f'torch=={torch.__version__.split(\"+\")[0]}')"],
    capture_output=True, text=True).stdout.strip()
print("pinning:", torch_pin)
Path("/kaggle/working/constraints.txt").write_text(torch_pin + "\n")
C = "-c /kaggle/working/constraints.txt"

for cmd in [
    f'pip install -q {C} "anomalib==1.1.1" scikit-image scikit-learn opencv-python-headless',
    f'pip install -q {C} lightning imgaug FrEIA einops timm kornia open_clip_torch',
    f'pip install -q {C} "matplotlib<3.10"',
    f'pip install -q {C} "numpy<2"',
]:
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True)

out = subprocess.run(
    [sys.executable, "-c",
     "import torch,numpy;print(torch.__version__,torch.cuda.is_available(),numpy.__version__)"],
    capture_output=True, text=True).stdout.strip()
print("after install ->", out)
assert "False" not in out, "torch lost CUDA — a CPU wheel got pulled in; check constraints.txt"
print("\n*** RESTART THE KERNEL NOW, then continue from cell 2. Do not re-run this cell. ***")

## 2 — Config

Re-run this cell after every kernel restart. It defines paths, the `sh()` helper and
`find_raw_root()`, so every later cell is independently runnable.

In [ ]:
import os, random, shutil, subprocess, sys, time
from pathlib import Path

WORK  = Path("/kaggle/working")
TMP   = Path("/kaggle/temp"); TMP.mkdir(parents=True, exist_ok=True)
INPUT = Path("/kaggle/input")

REPO     = WORK / "Unsupervised-Crack-Detection_CVPR"
REPO_URL = "https://github.com/Alirezanltv/Unsupervised-Crack-Detection_CVPR.git"
RAW      = TMP  / "deepcrack_raw"           # extracted DeepCrack.zip
DATA     = TMP  / "data" / "deepcrack"      # MVTec-style layout
RUNS     = WORK / "runs"
CKPT_DIR = WORK / "agdscae"                 # replaces /content/drive/MyDrive/agdscae

N_CALIB, SEED = 50, 0
SPLITS = ("train_img", "train_lab", "test_img", "test_lab")

# Colab also pulled SDNET2018 / CrackForest / BSDS500, but no cell below uses them.
FETCH_EXTRA_DATASETS = False


def sh(cmd, check=True, cwd=None):
    print(f"\n$ {cmd}", flush=True)
    rc = subprocess.run(cmd, shell=True, cwd=cwd).returncode
    if check and rc != 0:
        raise SystemExit(f"command failed (rc={rc}): {cmd}")
    return rc


def files(d):
    return sorted(p for p in Path(d).iterdir() if p.is_file() and not p.name.startswith("."))


def find_raw_root():
    """Dir under RAW that actually holds all four splits; ignores __MACOSX / nested copies."""
    cands = [p.parent for p in RAW.rglob("test_img")
             if p.is_dir() and "__MACOSX" not in p.parts
             and all((p.parent / s).is_dir() for s in SPLITS)]
    if not cands:
        raise SystemExit(f"no DeepCrack root under {RAW}; dirs: "
                         f"{[str(p.relative_to(RAW)) for p in RAW.rglob('*') if p.is_dir()][:20]}")
    return min(cands, key=lambda p: len(p.parts))


if REPO.is_dir():
    os.chdir(REPO)
print("cwd:", os.getcwd())

## 3 — Repo + GPU check

The Colab version cloned, then `rm -rf`'d the clone, then `cd`'d into it — it only worked
because the clone cell was re-run by hand. This is idempotent.

In [ ]:
if not (REPO / ".git").is_dir():
    shutil.rmtree(REPO, ignore_errors=True)
    sh(f"git clone --depth 1 {REPO_URL} {REPO}")
else:
    print("repo already present:", REPO)

os.chdir(REPO)
sh("nvidia-smi", check=False)
sh("python common/stats.py --selftest && python common/smoke_test.py")

## 4 — DeepCrack data

Prefers an attached Kaggle Dataset containing `DeepCrack.zip` over re-downloading every
session. Upload it once as a dataset and attach it via `Add Input`.

In [ ]:
zp = None
if INPUT.is_dir():
    zp = next(iter(INPUT.rglob("DeepCrack.zip")), None)
if zp is None:
    zp = TMP / "DeepCrack.zip"
    if not zp.exists():
        sh(f"wget -c -q -O {zp} https://raw.githubusercontent.com/yhlleo/DeepCrack/"
           f"master/dataset/DeepCrack.zip")
print("DeepCrack.zip ->", zp)

RAW.mkdir(parents=True, exist_ok=True)
sh(f"unzip -q -n {zp} -d {RAW}")
sh(f"find {RAW} -maxdepth 2 -type d")

if FETCH_EXTRA_DATASETS:
    sh(f"wget -c -q -O {TMP}/sdnet2018.zip "
       "'https://digitalcommons.usu.edu/context/all_datasets/article/1047/type/native/viewcontent'")
    sh(f"git clone --depth 1 https://github.com/cuilimeng/CrackForest-dataset.git {TMP}/CrackForest",
       check=False)
    sh(f"wget -c -q -O {TMP}/BSR_bsds500.tgz https://www2.eecs.berkeley.edu/Research/"
       "Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz")

## 5 — MVTec-style layout

`train/good`, `test/images`, `test/masks`, `calib` (50 held-out unlabeled train images).
RAW and DATA are both under `/kaggle/temp`, so hardlinks work — no duplication. If you
point RAW at `/kaggle/input` this silently degrades to a full copy.

In [ ]:
raw_root = find_raw_root()
print("root:", raw_root)

for sub in ("train/good", "test/images", "test/masks", "calib"):
    t = DATA / sub
    shutil.rmtree(t, ignore_errors=True)      # idempotent re-runs
    t.mkdir(parents=True)


def put(src, dst):
    try:
        (dst / src.name).hardlink_to(src)
    except (OSError, AttributeError):
        shutil.copy2(src, dst / src.name)


train = files(raw_root / "train_img")
if len(train) <= N_CALIB:
    raise SystemExit(f"only {len(train)} train images, need > {N_CALIB}")
calib = set(random.Random(SEED).sample(train, N_CALIB))
for p in train:
    put(p, DATA / ("calib" if p in calib else "train/good"))
for p in files(raw_root / "test_img"):
    put(p, DATA / "test/images")
for p in files(raw_root / "test_lab"):
    put(p, DATA / "test/masks")

print({s: len(files(DATA / s)) for s in ("train/good", "test/images", "test/masks", "calib")})
imgs  = {p.stem for p in files(DATA / "test/images")}
masks = {p.stem for p in files(DATA / "test/masks")}
print("img w/o mask:", len(imgs - masks), "| mask w/o img:", len(masks - imgs))

## 6 — Source patches

1. `eval_maps.py`: resize maps to ground-truth resolution instead of aborting on mismatch.
2. `run_baselines.py`: keep the whole test set in `predict` (`val_split_mode="same_as_test"`).

Both are no-ops if already applied.

In [ ]:
p = REPO / "common" / "eval_maps.py"
s = p.read_text()
if "resize_map" not in s:
    s = s.replace(
"""def standardize(raw: np.ndarray, mu: float, delta: float) -> np.ndarray:""",
"""def resize_map(raw: np.ndarray, shape) -> np.ndarray:
    from scipy.ndimage import zoom
    return zoom(raw, (shape[0] / raw.shape[0], shape[1] / raw.shape[1]), order=1)


def standardize(raw: np.ndarray, mu: float, delta: float) -> np.ndarray:""")
    s = s.replace(
"""        if raw.shape != gt.shape:
            raise SystemExit(f"shape mismatch {mp.name}: {raw.shape} vs {gt.shape}")""",
"""        if raw.shape != gt.shape:
            raw = resize_map(raw, gt.shape)""")
    p.write_text(s)

q = REPO / "p1_sota_baselines" / "run_baselines.py"
t = q.read_text()
if "same_as_test" not in t:
    t = t.replace("""        num_workers=2,
    )""", """        num_workers=2,
        val_split_mode="same_as_test",
    )""")
    q.write_text(t)
print("both patched")

## 7 — SOTA baselines (patchcore, padim)

In [ ]:
out = RUNS / "deepcrack"
shutil.rmtree(out, ignore_errors=True)
sh(f"python p1_sota_baselines/run_baselines.py --data {DATA} --out {out} "
   f"--models patchcore padim --seeds 0")

## 8 — Evaluate baselines

In [ ]:
out = RUNS / "deepcrack"
for m in ("patchcore", "padim"):
    print(m, "maps:", len(list((out / m / "s0" / "maps").glob("*"))))
    sh(f"python common/eval_maps.py --maps {out}/{m}/s0/maps "
       f"--masks {DATA}/test/masks --calib {out}/{m}/s0/calib "
       f"--out {RUNS}/{m}_deepcrack_s0.json")

## 9 — Splits for AGDSCAE

In [ ]:
raw_root = find_raw_root()
sh(f"python p0_reproduce/make_splits.py --name deepcrack "
   f"--images {raw_root}/train_img --test-images {raw_root}/test_img --calib {N_CALIB}")
sh("head -3 p0_reproduce/splits/deepcrack_train.txt && wc -l p0_reproduce/splits/deepcrack_*.txt")

## 10 — Smoke run (2 epochs, 2000 subset)

Times each stage and aborts the chain on the first non-zero return code. Use the reported
per-stage seconds to extrapolate whether the full run fits inside one GPU session.

In [ ]:
raw_root = find_raw_root()
smoke = TMP / "smoke"
shutil.rmtree(smoke, ignore_errors=True)

CMDS = [
    ("train",      f"python p0_reproduce/agdscae_ref.py train --raw-root {raw_root} "
                   f"--splits p0_reproduce/splits --name deepcrack --out {smoke} --seed 0 "
                   f"--stage-epochs 2 --source-subset 2000"),
    ("dump_test",  f"python p0_reproduce/agdscae_ref.py dump --ckpt {smoke}/ckpt_stage3.pt "
                   f"--images {DATA}/test/images --out {smoke}/maps"),
    ("dump_calib", f"python p0_reproduce/agdscae_ref.py dump --ckpt {smoke}/ckpt_stage3.pt "
                   f"--images {DATA}/calib --out {smoke}/calib"),
    ("eval",       f"python common/eval_maps.py --maps {smoke}/maps "
                   f"--masks {DATA}/test/masks --calib {smoke}/calib"),
]

times = {}
for name, cmd in CMDS:
    print(f"\n=== {name} ===", flush=True)
    t0 = time.perf_counter()
    rc = sh(cmd, check=False)
    times[name] = time.perf_counter() - t0
    print(f"--- {name}: {times[name]:.1f}s (rc={rc})", flush=True)
    if rc != 0:
        print("ABORT — later steps depend on this output.", file=sys.stderr)
        break

print("\n" + "\n".join(f"{k:<11} {v:7.1f}s" for k, v in times.items()))
print(f"{'TOTAL':<11} {sum(times.values()):7.1f}s")

## 11 — Full training (long)

No Google Drive on Kaggle — checkpoints go to `/kaggle/working`, which is persisted as
notebook output. To resume in a new session, attach this notebook's output as a dataset.

Run this via **Save Version -> Run All** (background execution) rather than interactively,
and confirm `agdscae_ref.py` writes per-stage checkpoints — if it only writes
`ckpt_stage3.pt` at the very end, a session timeout loses the entire run.

In [ ]:
raw_root = find_raw_root()
out = CKPT_DIR / "deepcrack_s0"
out.mkdir(parents=True, exist_ok=True)
sh(f"python p0_reproduce/agdscae_ref.py train --raw-root {raw_root} "
   f"--splits p0_reproduce/splits --name deepcrack --out {out} --seed 0 "
   f"--stage-epochs 100 --source-subset 20000")

In [ ]:
!pwd

## 12 — Dump + eval the full model

In [ ]:
out = CKPT_DIR / "deepcrack_s0"
sh(f"python p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt "
   f"--images {DATA}/test/images --out {out}/maps")
sh(f"python p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt "
   f"--images {DATA}/calib --out {out}/calib")
sh(f"python common/eval_maps.py --maps {out}/maps --masks {DATA}/test/masks "
   f"--calib {out}/calib --out {RUNS}/agdscae_deepcrack_s0.json")

In [ ]:
import subprocess, os
REPO = "/kaggle/working/Unsupervised-Crack-Detection_CVPR"   # adjust if elsewhere
DATA = "/kaggle/temp/data/deepcrack"                          # arranged: test/images, test/masks, calib
RAW  = "/kaggle/temp/deepcrack_raw"
os.chdir(REPO)

def pipeline(seed, gpu):
    out = f"/kaggle/working/agdscae/deepcrack_s{seed}"
    cmd = (f"python p0_reproduce/agdscae_ref.py train --raw-root {RAW} --splits p0_reproduce/splits "
           f"--name deepcrack --out {out} --seed {seed} --stage-epochs 50 --source-subset 20000 && "
           f"python p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt --images {DATA}/test/images --out {out}/maps && "
           f"python p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt --images {DATA}/calib --out {out}/calib && "
           f"python common/eval_maps.py --maps {out}/maps --masks {DATA}/test/masks --calib {out}/calib --out {out}/result.json")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"/kaggle/working/seed{seed}.log", "w")
    return subprocess.Popen(cmd, shell=True, env=env, stdout=log, stderr=subprocess.STDOUT)

procs = [pipeline(0, 0), pipeline(1, 1)]
for p in procs: p.wait()
for s in (0, 1):
    print(f"===== seed {s} ====="); print(open(f"/kaggle/working/agdscae/deepcrack_s{s}/result.json").read())


In [ ]:
import os, time, subprocess
from pathlib import Path
from IPython.display import FileLink

OUT   = Path("/kaggle/working")
STAMP = time.strftime("%Y%m%d_%H%M")
os.chdir(OUT)                                   # FileLink resolves relative to /kaggle/working

# what's actually there and how big
for s in (0, 1):
    d = OUT / f"agdscae/deepcrack_s{s}"
    print(f"seed {s}:", "MISSING" if not d.exists() else
          subprocess.run(f"du -sh {d}/* 2>/dev/null", shell=True,
                         capture_output=True, text=True).stdout.strip() or "empty")

# 1) light bundle: metrics + logs + checkpoints (no maps) — this is what you normally want
light = f"agdscae_results_{STAMP}.zip"
subprocess.run(
    f"zip -qr {light} agdscae -x 'agdscae/*/maps/*' 'agdscae/*/calib/*' && "
    f"zip -qj {light} seed0.log seed1.log", shell=True)

# 2) full bundle including anomaly maps — only if you need the raw maps off-platform
FULL = False
full = f"agdscae_full_{STAMP}.zip"
if FULL:
    subprocess.run(f"zip -qr {full} agdscae seed0.log seed1.log", shell=True)

for z in ([light] + ([full] if FULL else [])):
    print(f"{z}  {os.path.getsize(z)/1e6:.1f} MB")
    display(FileLink(z))

In [ ]:
#### Session 2 

In [1]:
!git clone https://github.com/Alirezanltv/Unsupervised-Crack-Detection_CVPR.git
%cd Unsupervised-Crack-Detection_CVPR
!pip install -q scikit-image scikit-learn


Cloning into 'Unsupervised-Crack-Detection_CVPR'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 115 (delta 31), reused 115 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 13.56 MiB | 47.56 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/kaggle/working/Unsupervised-Crack-Detection_CVPR


In [2]:
!mkdir -p /kaggle/temp
!wget -q https://raw.githubusercontent.com/yhlleo/DeepCrack/master/dataset/DeepCrack.zip && unzip -q DeepCrack.zip -d /kaggle/temp/deepcrack_raw
!python p0_reproduce/arrange_from_splits.py --raw-root /kaggle/temp/deepcrack_raw --splits p0_reproduce/splits --name deepcrack --dst /kaggle/temp/data/deepcrack --masks-dir /kaggle/temp/deepcrack_raw/test_lab


{'train/good': 250, 'calib': 50, 'test/images': 237, 'test/masks': 237}


In [ ]:
import subprocess, os, time

SEED, GPU = 4, 0
DATA, RAW = "/kaggle/temp/data/deepcrack", "/kaggle/temp/deepcrack_raw"
PY = "python -u"
PROGRESS_EVERY = 5.0

out = f"/kaggle/working/agdscae/deepcrack_s{SEED}"
log_path = f"/kaggle/working/seed{SEED}.log"

steps = [
    ("train",
     f"{PY} p0_reproduce/agdscae_ref.py train --raw-root {RAW} --splits p0_reproduce/splits "
     f"--name deepcrack --out {out} --seed {SEED} --stage-epochs 50 --source-subset 20000"),
    ("dump/test",
     f"{PY} p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt --images {DATA}/test/images --out {out}/maps"),
    ("dump/calib",
     f"{PY} p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt --images {DATA}/calib --out {out}/calib"),
    ("eval",
     f"{PY} common/eval_maps.py --maps {out}/maps --masks {DATA}/test/masks --calib {out}/calib --out {out}/result.json"),
    ("sweep",
     f"{PY} common/sweep_threshold.py --maps {out}/maps --masks {DATA}/test/masks --calib {out}/calib --out {out}/sweep.json"),
]
cmd = "set -e; " + " && ".join(f'echo "### STAGE {n} ###" && {c}' for n, c in steps)

env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPU), PYTHONUNBUFFERED="1")
p = subprocess.Popen(cmd, shell=True, env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)

t0, last_progress, buf = time.time(), 0.0, b""
with open(log_path, "w", buffering=1) as log:
    for chunk in iter(lambda: os.read(p.stdout.fileno(), 8192), b""):
        buf += chunk
        while True:
            i = min([x for x in (buf.find(b"\n"), buf.find(b"\r")) if x >= 0], default=-1)
            if i < 0:
                break
            line, sep, buf = buf[:i].decode("utf-8", "replace"), buf[i:i+1], buf[i+1:]
            if not line.strip():
                continue
            log.write(line + "\n")
            if sep == b"\r":                       # tqdm redraw
                if time.time() - last_progress < PROGRESS_EVERY:
                    continue
                last_progress = time.time()
            print(f"[{time.time()-t0:7.0f}s] {line}", flush=True)
    if buf.strip():
        print(f"[{time.time()-t0:7.0f}s] {buf.decode('utf-8','replace')}", flush=True)

rc = p.wait()
print(f"exit={rc} after {time.time()-t0:.0f}s → {out}", flush=True)

res = f"{out}/result.json"
print(f"===== seed {SEED} =====")
print(open(res).read() if os.path.exists(res) else f"MISSING {res} — check {log_path}")

[      0s] ### STAGE train ###
[    212s] [stage1 ep1/50] loss=0.0064
[    415s] [stage1 ep2/50] loss=0.0056
[    620s] [stage1 ep3/50] loss=0.0047


In [ ]:
import subprocess, os, threading, time, sys

DATA, RAW = "/kaggle/temp/data/deepcrack", "/kaggle/temp/deepcrack_raw"
PY = "python -u"                      # -u: unbuffered, otherwise no live output
PROGRESS_EVERY = 5.0                  # seconds; throttles tqdm \r spam

def build_cmd(seed, out):
    steps = [
        (f"{PY} p0_reproduce/agdscae_ref.py train --raw-root {RAW} --splits p0_reproduce/splits "
         f"--name deepcrack --out {out} --seed {seed} --stage-epochs 50 --source-subset 20000"),
        f"{PY} p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt --images {DATA}/test/images --out {out}/maps",
        f"{PY} p0_reproduce/agdscae_ref.py dump --ckpt {out}/ckpt_stage3.pt --images {DATA}/calib --out {out}/calib",
        f"{PY} common/eval_maps.py --maps {out}/maps --masks {DATA}/test/masks --calib {out}/calib --out {out}/result.json",
        f"{PY} common/sweep_threshold.py --maps {out}/maps --masks {DATA}/test/masks --calib {out}/calib --out {out}/sweep.json",
    ]
    names = ["train", "dump/test", "dump/calib", "eval", "sweep"]
    # echo markers so you can see which stage is live; set -e aborts the chain on failure
    body = " && ".join(f'echo "### STAGE {n} ###" && {c}' for n, c in zip(names, steps))
    return f"set -e; {body}"

def pump(proc, seed, logpath):
    """Mirror child output to notebook + logfile, line-tagged, tqdm throttled."""
    log = open(logpath, "w", buffering=1)
    buf, last_progress = b"", 0.0
    fd = proc.stdout.fileno()
    for chunk in iter(lambda: os.read(fd, 8192), b""):
        buf += chunk
        while True:
            i = min([p for p in (buf.find(b"\n"), buf.find(b"\r")) if p >= 0], default=-1)
            if i < 0:
                break
            line, sep, buf = buf[:i].decode("utf-8", "replace"), buf[i:i+1], buf[i+1:]
            if not line.strip():
                continue
            log.write(f"{line}\n")
            now = time.time()
            if sep == b"\r":                       # progress bar redraw
                if now - last_progress < PROGRESS_EVERY:
                    continue
                last_progress = now
            print(f"[s{seed}] {line}", flush=True)
    if buf.strip():
        print(f"[s{seed}] {buf.decode('utf-8','replace')}", flush=True)
    log.close()

def pipeline(seed, gpu):
    out = f"/kaggle/working/agdscae/deepcrack_s{seed}"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    p = subprocess.Popen(build_cmd(seed, out), shell=True, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    t = threading.Thread(target=pump, args=(p, seed, f"/kaggle/working/seed{seed}.log"), daemon=True)
    t.start()
    return p, t, out

jobs = [pipeline(2, 0), pipeline(3, 1)]
for p, t, out in jobs:
    rc = p.wait(); t.join()
    print(f"[s?] exit={rc} for {out}", flush=True)

for s in (2, 3):
    path = f"/kaggle/working/agdscae/deepcrack_s{s}/result.json"
    print(f"===== seed {s} =====")
    print(open(path).read() if os.path.exists(path) else f"MISSING {path} — check /kaggle/working/seed{s}.log")

In [3]:
import subprocess, os
DATA, RAW = "/kaggle/temp/data/deepcrack", "/kaggle/temp/deepcrack_raw"

# GPU 0: seed 4 (same chained pipeline you used for seeds 2/3)
p_seed = pipeline(4, 0)   # your existing pipeline() function

# GPU 1: PatchCore + PaDiM, 5 seeds, eval + sweep each
bl = (
  "pip install -q anomalib==1.1.1 && "
  f"python -u p1_sota_baselines/run_baselines.py --data {DATA} --out /kaggle/working/runs --models patchcore padim --seeds 0 1 2 3 4 && "
  + " && ".join(
      f"python common/eval_maps.py --maps /kaggle/working/runs/{m}/s{s}/maps --masks {DATA}/test/masks "
      f"--calib /kaggle/working/runs/{m}/s{s}/calib --out /kaggle/working/runs/{m}/s{s}/result.json && "
      f"python common/sweep_threshold.py --maps /kaggle/working/runs/{m}/s{s}/maps --masks {DATA}/test/masks "
      f"--calib /kaggle/working/runs/{m}/s{s}/calib --out /kaggle/working/runs/{m}/s{s}/sweep.json"
      for m in ("patchcore", "padim") for s in range(5))
)
p_bl = subprocess.Popen(bl, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
                        stdout=open("/kaggle/working/baselines.log", "w"), stderr=subprocess.STDOUT)
p_seed[0].wait(); p_bl.wait()


NameError: name 'pipeline' is not defined

In [ ]:
!find /kaggle/working -name "result.json" -o -name "sweep.json" | zip -q /kaggle/working/all_results.zip -@
!zip -qr /kaggle/working/checkpoints.zip /kaggle/working/agdscae -i "*.pt"
!ls -la /kaggle/working/*.zip


In [ ]:
!tail -20 /kaggle/working/baselines.log
!find /kaggle/working -name "*.zip" -o -name "result.json" | zip -q /kaggle/working/all_results.zip -@ 2>/dev/null; !zip -qr /kaggle/working/checkpoints.zip /kaggle/working/agdscae -i "*.pt"; !ls -la /kaggle/working/*.zip
